# 03 Model runnen

Hier worden de gegenereerde modellen gedraaid en de resultaten weggeschreven.

Dit script kan worden gedupliceerd om meerdere modellen parallel te draaien.

In [ ]:
import subprocess
import shutil
import pandas as pd
from pathlib import Path
import time

### Selecteer de gebieden en scenarios die gedraaid moeten worden

Ook wordt hier de periode waarover de modellen gerdraaid moeten worden aangegeven.

In [ ]:
# selectie_gebied = 0 # Oude IJssel
# selectie_gebied = 1 # West
# selectie_gebied = 2 # Centraal
# selectie_gebied = 3 # Oost

selectie_gebieden = [1,2,3]

scenarios = ["REF", "SCEN"]

# LONG RUN
# start_date = "2010-4-1"
# end_date = "2018-12-31"
# seizoenen = ["zomer", "winter"]
# date_range = pd.date_range(start_date, end_date, freq="2D")

# LONG TEST
start_date = "2015-07-1"
end_date = "2016-09-30"
seizoenen = ["zomer", "winter", "winter", "zomer"]
date_range = pd.date_range(start_date, end_date, freq="3MS")

# SMALL TEST
# start_date = "2012-4-1"
# end_date = "2012-4-12"
# seizoenen = ["zomer", "winter"]
# date_range = pd.date_range(start_date, end_date, freq="2D")

In [ ]:
# Nieuwe kD
# dir_model_basis = "..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\basis_aangepaste_kD"
# use_restart = False

# Nieuwe kD-restart
# dir_model_basis = "..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\basis_aangepaste_kD_restart"
# use_restart = True
# dir_model_restart = Path("..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\basis_aangepaste_kD")

# L2
# dir_model_basis = "..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\basis_L2"
# use_restart = False

# L2-restart
# dir_model_basis = "..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\basis_L2_restart"
# use_restart = True
# dir_model_restart = Path("..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\basis_L2")

# L2+infiltratie
# dir_model_basis = "..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\basis_L2_INF"
# use_restart = False

# L2+infiltratie-restart
# dir_model_basis = "..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\basis_L2_INF_restart"
# use_restart = True
# dir_model_restart = Path("..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\basis_L2_INF")

# L4+infiltratie+weerstand
# dir_model_basis = "..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\basis_L4_INF_W"
# use_restart = False

# L4+infiltratie+weerstand-restart
dir_model_basis = "..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\basis_L4_INF_W_restart"
use_restart = True
dir_model_restart = Path("..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\basis_L4_INF_W")

In [ ]:
simulations_total = pd.DataFrame()

for selectie_gebied in selectie_gebieden:
    for scenario in scenarios:

        simulaties = pd.DataFrame()
        simulaties["start_date"] = date_range
        simulaties["end_date"] = simulaties["start_date"].shift(-1)
        simulaties.loc[simulaties.index[-1],"end_date"] = pd.to_datetime(end_date)
        simulaties["seizoen"] = (seizoenen * 100)[:len(date_range)]
        simulaties["scenario"] = scenario
        simulaties["gebied"] = selectie_gebied
        simulaties["restart_in"] = [0] + [1] * len(simulaties.index[1:])

        simulaties["model_name"] = simulaties.apply(lambda x: f"rr_model__{pd.to_datetime(x.start_date).strftime('%Y_%m_%d')}__{pd.to_datetime(x.end_date).strftime('%Y_%m_%d')}", axis=1)
        simulations_total = pd.concat([simulations_total, simulaties])

In [ ]:
first_simulations = simulations_total.groupby(["gebied", "scenario"]).first()
last_simulations = simulations_total.groupby(["gebied", "scenario"]).last()

if use_restart:
    simulations_total["restart_in"] = 1
    for selectie_gebied in selectie_gebieden:
        for scenario in scenarios:
            last_simulation = last_simulations.loc[(selectie_gebied, scenario)]
            path_rr_restart_old = Path(
                dir_model_restart, 
                f"gebied_{selectie_gebied}", 
                scenario, 
                last_simulation.model_name,
                "rr",
                "RSRR_OUT"
            )
            first_simulation = first_simulations.loc[(selectie_gebied, scenario)]
            path_rr_restart_new = Path(
                dir_model_basis, 
                f"gebied_{selectie_gebied}", 
                scenario, 
                first_simulation.model_name,
                "rr",
                "RSRR_IN"
            )
            path_rr_restart_new.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(path_rr_restart_old, path_rr_restart_new)

simulations_total

### Hier worden de geselcteerde modellen gedraaid

Ook worden restartbestanden doorgegeven, zodat opeenvolgende modelruns op elkaar kunnen aansluiten.

In [ ]:
def calculate_simulation_train(simulations_group):
    for index, simulatie in simulations_group.iterrows():
        print("START: " + str(simulatie.gebied) + " - " + simulatie.scenario + " - " + simulatie.model_name)
        dir_model = Path(dir_model_basis, f"gebied_{simulatie.gebied}", simulatie.scenario)
        
        # copy restart-file from previous run
        if index > 0:
            unpaved_restart_out = Path(dir_model, simulaties.loc[index-1, "model_name"], "rr", "RSRR_OUT")
            unpaved_restart_in = Path(dir_model, simulaties.loc[index, "model_name"], "rr", "RSRR_IN")
            shutil.copy(unpaved_restart_out, unpaved_restart_in)

        workdir = Path(dir_model, simulaties.loc[index, "model_name"])
        batfile = workdir / "run.bat"
        result = subprocess.run(
            ["cmd.exe", "/c", str(batfile.resolve())],
            cwd=str(workdir),
            capture_output=True,
            text=True
        )

        print(str(simulatie.gebied) + " - " + simulatie.scenario + " - " + simulatie.model_name)
        print("STDOUT:")
        print(result.stdout)
        print("STDERR:")
        print(result.stderr)
        print("FINISH: " + str(simulatie.gebied) + " - " + simulatie.scenario + " - " + simulatie.model_name)

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

async def main():
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor() as pool:
        results = await asyncio.gather(
            *(
                loop.run_in_executor(pool, calculate_simulation_train, simulations_group.copy())
                for _, simulations_group in simulations_total.groupby(["gebied", "scenario"])
            )
        )
    return results

results = await main()